In [1]:
her2st_folder = '../data/her2st/data'
stnet_folder = '../data/ST/BC'
skin_folder = '../data/skin'
pcw_folder = '../data/ST-images/PCW'
mouse_folder = '../data/ST-images/MOUSE'

her2st_imgs_folder = her2st_folder + '/ST-imgs'
her2st_st_postion = her2st_folder + '/ST-spotfiles'
her2st_gene_expression = her2st_folder + '/ST-cnts'
print(her2st_imgs_folder)
print(her2st_st_postion)
print(her2st_gene_expression)

stnet_imgs_folder = stnet_folder + '/ST-imgs'
stnet_st_postion = stnet_folder + '/ST-spotfiles'
stnet_gene_expression = stnet_folder + '/ST-cnts'
print(stnet_imgs_folder)
print(stnet_st_postion)
print(stnet_gene_expression)

skin_imgs_folder = skin_folder + '/ST-imgs'
skin_st_postion = skin_folder + '/ST-spotfiles'
skin_gene_expression = skin_folder + '/ST-cnts'
print(skin_imgs_folder)
print(skin_st_postion)
print(skin_gene_expression)

pcw_imgs_folder =  pcw_folder + '/ST-imgs'
pcw_st_postion = pcw_folder + '/ST-spotfiles'
pcw_gene_expression = pcw_folder + '/ST-cnts'
print(pcw_imgs_folder)
print(pcw_st_postion)
print(pcw_gene_expression)

mouse_imgs_folder =  mouse_folder + '/ST-imgs'
mouse_st_postion = mouse_folder + '/ST-spotfiles'
mouse_gene_expression = mouse_folder + '/ST-cnts'
print(mouse_imgs_folder)
print(mouse_st_postion)
print(mouse_gene_expression)

In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def display_image_info_and_show(image_path, tsv_path, re_width = 224, line_width=0.2):
    # Open the image file
    with Image.open(image_path) as img:
        # Get image dimensions
        width, height = img.size
        print(f"Image size: {width} x {height} pixels")
        
        data = pd.read_csv(tsv_path, sep='\t')
        pixel_x = data['pixel_x']
        pixel_y = data['pixel_y']
        # fig, ax = plt.subplots()
        fig, ax = plt.subplots(figsize=(10, 10), dpi=150)
        ax.imshow(img)
        
        spot_num = 0
        
        for x, y in zip(pixel_x, pixel_y):
            spot_num = spot_num + 1
            rect = patches.Rectangle((x - re_width/2, y - re_width/2), re_width, re_width,
                                     linewidth=line_width,
                                     edgecolor='r',
                                     facecolor='none')
            ax.add_patch(rect)
        plt.title('File '+ str(image_path.split('/')[-1]) +'   Spot num = ' + str(spot_num), fontsize=16)
        plt.axis('off') 
        plt.show()
        

display_image_info_and_show(her2st_imgs_folder+'/A/A1/A1.jpg', her2st_st_postion + '/A1_selection.tsv')
display_image_info_and_show(her2st_imgs_folder+'/A/A2/A2.jpg', her2st_st_postion + '/A2_selection.tsv')
display_image_info_and_show(her2st_imgs_folder+'/A/A3/A3.jpg', her2st_st_postion + '/A3_selection.tsv')
display_image_info_and_show(her2st_imgs_folder+'/A/A4/A4.jpg', her2st_st_postion + '/A4_selection.tsv')
display_image_info_and_show(her2st_imgs_folder+'/A/A5/A5.jpg', her2st_st_postion + '/A5_selection.tsv')
display_image_info_and_show(her2st_imgs_folder+'/A/A6/A6.jpg', her2st_st_postion + '/A6_selection.tsv')

In [4]:
registration_xfeat_her2st_dir = []

registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/A/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/B/registration_xfeat')
registration_xfeat_her2st_dir

In [ ]:
import os
import numpy as np

def plot_squares_on_images(image_path_A1, image_path_A2, coord_A1, coord_A2):
    img_A1 = Image.open(image_path_A1)
    img_A2 = Image.open(image_path_A2)
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    axes[0].imshow(img_A1)
    rect_A1 = patches.Rectangle((coord_A1[0] - 112, coord_A1[1] - 112), 
                                224,
                                224,
                                linewidth=3,
                                edgecolor='r',
                                facecolor='none')
    axes[0].add_patch(rect_A1)
    axes[0].set_title('Image src with Spot')

    axes[1].imshow(img_A2)
    rect_A2 = patches.Rectangle((coord_A2[0] - 112, coord_A2[1] - 112),
                                224,
                                224,
                                linewidth=3,
                                edgecolor='b',
                                facecolor='none')
    axes[1].add_patch(rect_A2)
    axes[1].set_title('Image dst with Transformed Spot')

    plt.show()

def plot_registration(spot_num):
    for i in registration_xfeat_her2st_dir:
        for j in os.listdir(i):
            print(j)
            src = j.split('_')[2]
            dst = j.split('_')[0]
            for k in os.listdir(os.path.join(i,j)):
                if k.endswith('npy'):
                    print(k)
                    print(f"From {dst} to {src} :")
                    file_path = os.path.join(i,j,k)
                    print(file_path)
                    aff = np.load(file_path)
                    print(aff)
                    src_spots = pd.read_csv(her2st_st_postion + '/' + str(src) +'_selection.tsv', sep='\t')
                    dst_spots = pd.read_csv(her2st_st_postion + '/' + str(dst) +'_selection.tsv', sep='\t')
                    pixel_x_value = src_spots['pixel_x'].iloc[spot_num]
                    pixel_y_value = src_spots['pixel_y'].iloc[spot_num]
                    print(pixel_x_value)
                    print(pixel_y_value)
                    aff_homogeneous = np.vstack([aff, [0, 0, 1]])
                    inv_aff_homogeneous = np.linalg.inv(aff_homogeneous)
                    point_on_A1_homogeneous = np.array([pixel_x_value, pixel_y_value, 1])
                    original_point_in_A2_homogeneous = inv_aff_homogeneous @ point_on_A1_homogeneous
                    original_point_in_A2 = original_point_in_A2_homogeneous[:2] / original_point_in_A2_homogeneous[2]
                    print(f"Original position in {dst}: {original_point_in_A2}")
                    plot_squares_on_images(
                        her2st_imgs_folder+f'/A/{src}/{src}.jpg', 
                        her2st_imgs_folder+f'/A/{dst}/{dst}.jpg', 
                        (pixel_x_value, pixel_y_value),   
                        (original_point_in_A2[0], original_point_in_A2[1])
                    )
                else:
                    continue

for i in range(2):
    plot_registration(i)

In [9]:
for i in range(4):
    plot_registration(i+100)

In [ ]:
registration_xfeat_her2st_dir = []

registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/A/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/B/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/C/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/D/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/E/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/F/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/G/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/H/registration_xfeat')

print(registration_xfeat_her2st_dir)
from PIL import ImageDraw,Image
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_rectangles_on_images(all_img_files, what_can_i_say, rect_size=224):
    num_colors = len(what_can_i_say)
    colors = plt.cm.get_cmap('tab10', num_colors) 
    for i, img_file in enumerate(all_img_files):
        img = Image.open(img_file)
        draw = ImageDraw.Draw(img)
        for j, ((src_x, src_y), dsts_aff) in enumerate(what_can_i_say):
            if i == 0:
                x, y = src_x, src_y
            else:
                x, y = dsts_aff[i - 1]
            rect_x1 = x - rect_size / 2
            rect_y1 = y - rect_size / 2
            rect_x2 = x + rect_size / 2
            rect_y2 = y + rect_size / 2
            color_rgb = tuple(int(255 * c) for c in colors(j)[:3])
            draw.rectangle([rect_x1, rect_y1, rect_x2, rect_y2], outline=color_rgb, width=20)
        plt.figure(figsize=(10, 10))
        plt.imshow(img)
        plt.title(f"Image {i+1}")
        plt.axis('off')
        plt.show()

def plot_aff_spots(src_img_file,src_spot_file,trans_aff,dstes_img_files,num_spots=10):
    print("--------------------  Lets polt something ------------------------")
    img_nums = len(dstes_img_files)+1
    assert(img_nums-1 == len(dstes_img_files))
    assert(img_nums-1 == len(trans_aff))
    src_spots = pd.read_csv(src_spot_file, sep='\t')
    what_can_i_say = []
    for i in range(num_spots):
        src_x = src_spots['pixel_x'].iloc[i]
        src_y = src_spots['pixel_y'].iloc[i]
        current_coords = np.array([src_x, src_y, 1])
        dsts_aff = []
        for j in range(len(trans_aff)):
            aff_matrix = np.load(trans_aff[j])
            aff_matrix = np.vstack([aff_matrix, [0, 0, 1]])
            inv_aff_matrix = np.linalg.inv(aff_matrix)
            transformed_coord = inv_aff_matrix @ current_coords
            dst_x = transformed_coord[0]
            dst_y = transformed_coord[1]
            current_coords = transformed_coord
            dsts_aff.append((dst_x,dst_y))
        what_can_i_say.append(((src_x,src_y),dsts_aff))
    assert(len(what_can_i_say)==num_spots)
    all_img_files = [src_img_file] + dstes_img_files
    draw_rectangles_on_images(all_img_files, what_can_i_say)
    
sample_spots_nums = [348,290,175,300,585,690,440,610]
index = 0

Image.MAX_IMAGE_PIXELS = None 
for sample in registration_xfeat_her2st_dir:
    print(sample)
    orders = []
    trans_aff = []
    for order in os.listdir(sample):
        orders.append(order)
        for file in os.listdir(os.path.join(sample,order)):
            if file.endswith('.npy'):
                trans_aff.append(os.path.join(sample,order,file))
    print(orders)
    print(trans_aff)
    assert(len(orders) == len(trans_aff))
    nums = len(orders)
    src_img = orders[0].split('_')[2]
    print(src_img)
    dstes = []
    for i in orders:
        dstes.append(i.split('_')[0])
    print(dstes)
    src_img_file = os.path.join(os.path.dirname(sample),src_img,f'{src_img}.jpg')
    src_spot_file = os.path.join(her2st_st_postion,f'{src_img}_selection.tsv')
    dstes_img_files = []
    for i in dstes:
        dstes_img_files.append(os.path.join(os.path.dirname(sample),i,f'{i}.jpg'))
    print(dstes_img_files)
    plot_aff_spots(src_img_file,src_spot_file,trans_aff,dstes_img_files,num_spots=sample_spots_nums[index])
    index += 1
    

In [ ]:
print(registration_xfeat_her2st_dir)

import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

Image.MAX_IMAGE_PIXELS = None

sample_spots_nums = [348,290,175,300,585,690,440,610]
index = 0

def save_cropped_images_with_high_dpi(cropped_image, output_file, dpi=300):
    fig, ax = plt.subplots(figsize=(cropped_image.size[0] / dpi, cropped_image.size[1] / dpi))
    ax.imshow(cropped_image)
    ax.axis('off')  
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0) 

    fig.savefig(output_file, dpi=dpi, bbox_inches='tight', pad_inches=0)
    plt.close(fig)  

def transform_coordinates(src_x, src_y, aff_matrix):
    src_coords = np.array([src_x, src_y, 1])
    if aff_matrix.shape == (2, 3):
        aff_matrix = np.vstack([aff_matrix, [0, 0, 1]])
    inv_aff_matrix = np.linalg.inv(aff_matrix)
    dst_coords = np.dot(inv_aff_matrix, src_coords)
    return dst_coords[0], dst_coords[1]

def crop_image(image, center_x, center_y, size=224):
    left = int(center_x - size / 2)
    top = int(center_y - size / 2)
    right = left + size
    bottom = top + size
    cropped_image = image.crop((left, top, right, bottom))
    return cropped_image

for sample in registration_xfeat_her2st_dir:
    
    print(sample)
    orders = []
    trans_aff = []
    for order in os.listdir(sample):
        if order.endswith('.csv'):
            continue
        orders.append(order)
        
        for file in os.listdir(os.path.join(sample,order)):
            if file.endswith('.npy'):
                trans_aff.append(os.path.join(sample,order,file))
    print(orders)
    print(trans_aff)
    assert(len(orders) == len(trans_aff))
    nums = len(orders)
    src_img = orders[0].split('_')[2]
    print(src_img)
    dstes = []
    for i in orders:
        dstes.append(i.split('_')[0])
    print(dstes)
    src_img_file = os.path.join(os.path.dirname(sample),src_img,f'{src_img}.jpg')
    src_spot_file = os.path.join(her2st_st_postion,f'{src_img}_selection.tsv')
    dstes_img_files = []
    for i in dstes:
        dstes_img_files.append(os.path.join(os.path.dirname(sample),i,f'{i}.jpg'))
    print(dstes_img_files)

    src_spots = pd.read_csv(src_spot_file, sep='\t')
    for spot_index in range(len(src_spots)):
        src_x = src_spots['pixel_x'].iloc[spot_index]
        src_y = src_spots['pixel_y'].iloc[spot_index]
        current_x, current_y = src_x, src_y
        
        fig, axes = plt.subplots(1, len(dstes_img_files) + 1, figsize=(20, 5))
        fig.suptitle(f"Spot {spot_index} Cropped Images", fontsize=16)
        
        try:
            cropped_image_a1 = crop_image(Image.open(src_img_file), src_x, src_y)
            output_dir_a1 = os.path.join(os.path.dirname(src_img_file), 'cropped_images')
            os.makedirs(output_dir_a1, exist_ok=True)
            output_file_a1 = os.path.join(output_dir_a1, f'spot_{spot_index}_to_{src_img}.png')
            print(f"Cropped image saved to: {output_file_a1}")
            save_cropped_images_with_high_dpi(cropped_image_a1, output_file_a1, dpi=600)
            print(f"Cropped image saved to: {output_file_a1}")
            axes[0].imshow(cropped_image_a1)
            axes[0].set_title(f"{src_img}")
            axes[0].axis('off')
        except Exception as e:
            print(f"Failed to crop image for Spot {spot_index} in {src_img}: {e}")
        
        for j, aff_file in enumerate(trans_aff):
            aff_matrix = np.load(aff_file)
            current_x, current_y = transform_coordinates(current_x, current_y, aff_matrix)
            dst_img_file = dstes_img_files[j]
            dst_image = Image.open(dst_img_file)
            try:
                cropped_image = crop_image(dst_image, current_x, current_y)
            except Exception as e:
                print(f"Failed to crop image for Spot {spot_index} in {dstes[j]}: {e}")
                continue
            output_dir = os.path.join(os.path.dirname(dst_img_file), 'cropped_images')
            os.makedirs(output_dir, exist_ok=True)
            output_file = os.path.join(output_dir, f'spot_{spot_index}_to_{dstes[j]}.png')
            cropped_image.save(output_file)
            print(f"Cropped image saved to: {output_file}")
            axes[j + 1].imshow(cropped_image)
            axes[j + 1].set_title(f"{dstes[j]}")
            axes[j + 1].axis('off')
        plt.tight_layout()
        plt.show()
        if index == 3:
            break
    index += 1
    

In [ ]:
print(registration_xfeat_her2st_dir)

import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

block_size = 224

Image.MAX_IMAGE_PIXELS = None

sample_spots_nums = [348,290,175,300,585,690,440,610]
index = 0

def transform_coordinates(src_x, src_y, aff_matrix):
    src_coords = np.array([src_x, src_y, 1])
    if aff_matrix.shape == (2, 3):
        aff_matrix = np.vstack([aff_matrix, [0, 0, 1]])
    inv_aff_matrix = np.linalg.inv(aff_matrix)
    dst_coords = np.dot(inv_aff_matrix, src_coords)
    return dst_coords[0], dst_coords[1]
def crop_image(image, center_x, center_y, size=224):
    left = int(center_x - size / 2)
    top = int(center_y - size / 2)
    right = left + size
    bottom = top + size
    cropped_image = image.crop((left, top, right, bottom))
    return cropped_image

Image.MAX_IMAGE_PIXELS = None 
for sample in registration_xfeat_her2st_dir:
    print(sample)
    orders = []
    trans_aff = []
    for order in os.listdir(sample):
        orders.append(order)
        for file in os.listdir(os.path.join(sample,order)):
            if file.endswith('.npy'):
                trans_aff.append(os.path.join(sample,order,file))
    print(orders)
    print(trans_aff)
    assert(len(orders) == len(trans_aff))
    nums = len(orders)
    src_img = orders[0].split('_')[2]
    print(src_img)
    dstes = []
    for i in orders:
        dstes.append(i.split('_')[0])
    print(dstes)
    src_img_file = os.path.join(os.path.dirname(sample),src_img,f'{src_img}.jpg')
    src_spot_file = os.path.join(her2st_st_postion,f'{src_img}_selection.tsv')
    dstes_img_files = []
    for i in dstes:
        dstes_img_files.append(os.path.join(os.path.dirname(sample),i,f'{i}.jpg'))
    print(dstes_img_files)
     
    src_spots = pd.read_csv(src_spot_file, sep='\t')
    for spot_index in range(len(src_spots)):
        src_x = src_spots['pixel_x'].iloc[spot_index]
        src_y = src_spots['pixel_y'].iloc[spot_index]
        current_x, current_y = src_x, src_y
        
        for j, dst in enumerate(dstes):
            dst_spot_file = os.path.join(her2st_st_postion, f'{dst}_selection.tsv')
            dst_spots = pd.read_csv(dst_spot_file, sep='\t')
            
            aff_matrix = np.load(trans_aff[j])
            
            current_x, current_y = transform_coordinates(current_x, current_y, aff_matrix)
            
            block_x = int(current_x // block_size)
            block_y = int(current_y // block_size)
            
            matched_spot = dst_spots[
                (dst_spots['x'] == block_x) & (dst_spots['y'] == block_y)
            ]
            if not matched_spot.empty:
                dst_img_file = dstes_img_files[j]
                dst_image = Image.open(dst_img_file)
                try:
                    cropped_image = crop_image(dst_image, current_x, current_y)
                    output_dir = os.path.join(os.path.dirname(dst_img_file), 'cropped_images')
                    os.makedirs(output_dir, exist_ok=True)
                    output_file = os.path.join(output_dir, f'spot_{spot_index}_to_{dst}.png')
                    cropped_image.save(output_file)
                    print(f"Cropped image saved to: {output_file}")
                except Exception as e:
                    print(f"Failed to crop image for Spot {spot_index} in {dst}: {e}")
            else:
                print(f"No matching spot found for Spot {spot_index} in {dst} at block ({block_x}, {block_y})")
            break
    index += 1
    break
    

In [ ]:
registration_xfeat_her2st_dir = []
# skin_imgs_folder = skin_folder + '/ST-imgs'
# skin_st_postion = skin_folder + '/ST-spotfiles'
# skin_gene_expression = skin_folder + '/ST-cnts'
# print(skin_imgs_folder)
# print(skin_st_postion)
# print(skin_gene_expression)
registration_xfeat_her2st_dir.append(skin_imgs_folder + '/A/registration_xfeat')
# registration_xfeat_her2st_dir.append(skin_imgs_folder + '/B/registration_xfeat')
# registration_xfeat_her2st_dir.append(skin_imgs_folder + '/C/registration_xfeat')
# registration_xfeat_her2st_dir.append(skin_imgs_folder + '/D/registration_xfeat')
# registration_xfeat_her2st_dir.append(skin_imgs_folder + '/E/registration_xfeat')
# registration_xfeat_her2st_dir.append(skin_imgs_folder + '/F/registration_xfeat')

import os
import numpy as np
import pandas as pd

block_size = 224

def transform_coordinates(src_x, src_y, aff_matrix):
    src_coords = np.array([src_x, src_y, 1])
    if aff_matrix.shape == (2, 3):
        aff_matrix = np.vstack([aff_matrix, [0, 0, 1]])
    inv_aff_matrix = np.linalg.inv(aff_matrix)
    dst_coords = np.dot(inv_aff_matrix, src_coords)
    return dst_coords[0], dst_coords[1]

def get_surrounding_spots(x, y, spots_df):
    surrounding_spots = []
    matched_spot = spots_df[(spots_df['x'] == x) & (spots_df['y'] == y)]
    if not matched_spot.empty:
        surrounding_spots.append(matched_spot.index[0]) 
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0:
                continue  
            matched_spot = spots_df[(spots_df['x'] == x + dx) & (spots_df['y'] == y + dy)]
            if not matched_spot.empty:
                surrounding_spots.append(matched_spot.index[0]) 
    return surrounding_spots

def save_mapping_to_csv(mapping_results, output_file):
    df = pd.DataFrame(mapping_results)
    df.to_csv(output_file, index=False)
    print(f"Mapping results saved to: {output_file}")

def find_nearest_spot(dst_x, dst_y, spots_df, image_name):
    distances = np.sqrt((spots_df['pixel_x'] - dst_x)**2 + (spots_df['pixel_y'] - dst_y)**2)
    nearest_index = distances.idxmin()
    nearest_pixel_x = spots_df.loc[nearest_index]['pixel_x']
    nearest_pixel_y = spots_df.loc[nearest_index]['pixel_y']
    if abs(nearest_pixel_x - dst_x) > 112 and abs(nearest_pixel_y - dst_y) > 112 and nearest_index > 180:
        print(f"Image '{image_name}': Nearest spot is out of range: ({nearest_pixel_x}, {nearest_pixel_y})")
        return 0, 0
    nearest_block_x = spots_df.loc[nearest_index]['x']
    nearest_block_y = spots_df.loc[nearest_index]['y']
    return nearest_block_x, nearest_block_y

def get_subdirectories(directory):
    return [name for name in os.listdir(directory) if os.path.isdir(os.path.join(directory, name))]

for sample_dir in registration_xfeat_her2st_dir:
    print(f"Processing sample: {sample_dir}")
    sample_name = os.path.basename(os.path.dirname(sample_dir))
    print(f"Sample name: {sample_name}")
    mapping_results = []
    orders = []
    trans_aff = []
    subdirs = get_subdirectories(sample_dir)
    for order in subdirs:
        orders.append(order)
        for file in os.listdir(os.path.join(sample_dir, order)):
            if file.endswith('.npy'):
                trans_aff.append(os.path.join(sample_dir, order, file))
    print(f"Orders: {orders}")
    print(f"Affine files: {trans_aff}")
    assert len(orders) == len(trans_aff)
    src_img = orders[0].split('_')[2]
    print(f"Source image: {src_img}")
    dstes = [order.split('_')[0] for order in orders]
    print(f"Target images: {dstes}")
    src_spot_file = os.path.join(her2st_st_postion, f'{src_img}_selection.tsv')
    src_spots = pd.read_csv(src_spot_file, sep='\t')
    dst_spots_files = [os.path.join(her2st_st_postion, f'{dst}_selection.tsv') for dst in dstes]
    dst_spots_dfs = [pd.read_csv(file, sep='\t') for file in dst_spots_files]
    for spot_index in range(len(src_spots)):
        src_x = src_spots['pixel_x'].iloc[spot_index]
        src_y = src_spots['pixel_y'].iloc[spot_index]
        src_block_x = src_spots['x'].iloc[spot_index]
        src_block_y = src_spots['y'].iloc[spot_index]
        src_surrounding_spots = get_surrounding_spots(src_block_x, src_block_y, src_spots)
        spot_mapping = {f'{src_img}_Spots': src_surrounding_spots}
        current_x, current_y = src_x, src_y
        for j, dst in enumerate(dstes):
            selc_path = os.path.join(her2st_st_postion,dst+'_selection.tsv')
            aff_matrix = np.load(trans_aff[j])
            dst_x, dst_y = transform_coordinates(current_x, current_y, aff_matrix)
            current_x, current_y = dst_x, dst_y
            dst_sport_dfs = pd.read_csv(selc_path ,sep='\t')
            nearest_block_x,nearest_block_y = find_nearest_spot(dst_x,dst_y,dst_sport_dfs,dst)
            if nearest_block_x != 0 and nearest_block_y != 0:
                dst_block_x = nearest_block_x
                dst_block_y = nearest_block_y
                dst_surrounding_spots = get_surrounding_spots(dst_block_x, dst_block_y, dst_spots_dfs[j])
                spot_mapping[f'{dst}_Spots'] = dst_surrounding_spots
            else:
                spot_mapping[f'{dst}_Spots'] = []
        mapping_results.append(spot_mapping)
    output_file = os.path.join(sample_dir, f'{sample_name}_spot_mapping.csv')
    save_mapping_to_csv(mapping_results, output_file)
    mapping_results = []

In [ ]:
registration_xfeat_her2st_dir = []

# registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/A/registration_xfeat')
# registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/B/registration_xfeat')
# registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/C/registration_xfeat')
# registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/D/registration_xfeat')
# registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/E/registration_xfeat')
# registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/F/registration_xfeat')
# registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/G/registration_xfeat')
# registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/H/registration_xfeat')

print(registration_xfeat_her2st_dir)
from PIL import ImageDraw,Image
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_rectangles_on_images(all_img_files, what_can_i_say, rect_size=224):
    num_colors = len(what_can_i_say)
    colors = plt.cm.get_cmap('tab10', num_colors) 
    for i, img_file in enumerate(all_img_files):
        img = Image.open(img_file)
        draw = ImageDraw.Draw(img)
        for j, ((src_x, src_y), dsts_aff) in enumerate(what_can_i_say):
            if i == 0:
                x, y = src_x, src_y
            else:
                x, y = dsts_aff[i - 1]
            rect_x1 = x - rect_size / 2
            rect_y1 = y - rect_size / 2
            rect_x2 = x + rect_size / 2
            rect_y2 = y + rect_size / 2
            color_rgb = tuple(int(255 * c) for c in colors(j)[:3])
            draw.rectangle([rect_x1, rect_y1, rect_x2, rect_y2], outline=color_rgb, width=20)
        plt.figure(figsize=(10, 10))
        plt.imshow(img)
        plt.title(f"Image {i+1}")
        plt.axis('off')
        plt.show()

def plot_aff_spots(src_img_file,src_spot_file,trans_aff,dstes_img_files,num_spots=10):
    print("--------------------  Lets polt something ------------------------")
    img_nums = len(dstes_img_files)+1
    assert(img_nums-1 == len(dstes_img_files))
    assert(img_nums-1 == len(trans_aff))
    src_spots = pd.read_csv(src_spot_file, sep='\t')
    what_can_i_say = []
    for i in range(num_spots):
        src_x = src_spots['pixel_x'].iloc[i]
        src_y = src_spots['pixel_y'].iloc[i]
        current_coords = np.array([src_x, src_y, 1])
        dsts_aff = []
        for j in range(len(trans_aff)):
            aff_matrix = np.load(trans_aff[j])
            aff_matrix = np.vstack([aff_matrix, [0, 0, 1]])
            inv_aff_matrix = np.linalg.inv(aff_matrix)
            transformed_coord = inv_aff_matrix @ current_coords
            dst_x = transformed_coord[0]
            dst_y = transformed_coord[1]
            current_coords = transformed_coord
            dsts_aff.append((dst_x,dst_y))
        what_can_i_say.append(((src_x,src_y),dsts_aff))
    assert(len(what_can_i_say)==num_spots)
    all_img_files = [src_img_file] + dstes_img_files
    draw_rectangles_on_images(all_img_files, what_can_i_say)
    
sample_spots_nums = [348,290,175,300,585,690,440,610]
index = 0

Image.MAX_IMAGE_PIXELS = None 
for sample in registration_xfeat_her2st_dir:
    print(sample)
    orders = []
    trans_aff = []
    for order in os.listdir(sample):
        orders.append(order)
        for file in os.listdir(os.path.join(sample,order)):
            if file.endswith('.npy'):
                trans_aff.append(os.path.join(sample,order,file))
    print(orders)
    print(trans_aff)
    assert(len(orders) == len(trans_aff))
    nums = len(orders)
    src_img = orders[0].split('_')[2]
    print(src_img)
    dstes = []
    for i in orders:
        dstes.append(i.split('_')[0])
    print(dstes)
    src_img_file = os.path.join(os.path.dirname(sample),src_img,f'{src_img}.jpg')
    src_spot_file = os.path.join(her2st_st_postion,f'{src_img}_selection.tsv')
    dstes_img_files = []
    for i in dstes:
        dstes_img_files.append(os.path.join(os.path.dirname(sample),i,f'{i}.jpg'))
    print(dstes_img_files)
    plot_aff_spots(src_img_file,src_spot_file,trans_aff,dstes_img_files,num_spots=sample_spots_nums[index])
    index += 1
    

In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def display_image_info_and_show(image_path, csv_path, re_width = 224, line_width=1):
    # Open the image file
    with Image.open(image_path) as img:
        width, height = img.size
        print(f"Image size: {width} x {height} pixels")
        
        data = pd.read_csv(csv_path)
        pixel_x = data['pixel_x']
        pixel_y = data['pixel_y']
        fig, ax = plt.subplots(figsize=(10, 10), dpi=150)
        ax.imshow(img)
        
        spot_num = 0
        
        for x, y in zip(pixel_x, pixel_y):
            spot_num = spot_num + 1
            rect = patches.Rectangle((x - re_width/2, y - re_width/2), re_width, re_width,
                                     linewidth=line_width,
                                     edgecolor='white',
                                     facecolor='none')
            ax.add_patch(rect)
        plt.title('File '+ str(image_path.split('/')[-1]) +'   Spot num = ' + str(spot_num), fontsize=16)
        plt.axis('off')  
        plt.show()
        

# display_image_info_and_show(stnet_imgs_folder+'/A/A1.jpg', stnet_st_postion + '/A1.csv')
# display_image_info_and_show(stnet_imgs_folder+'/A/A2.jpg', stnet_st_postion + '/A2.csv')
# display_image_info_and_show(stnet_imgs_folder+'/A/A3.jpg', stnet_st_postion + '/A3.csv')
# display_image_info_and_show(stnet_imgs_folder+'/B/B1.jpg', stnet_st_postion + '/B1.csv')
# display_image_info_and_show(stnet_imgs_folder+'/B/B2.jpg', stnet_st_postion + '/B2.csv')
# display_image_info_and_show(stnet_imgs_folder+'/B/B3.jpg', stnet_st_postion + '/B3.csv')

display_image_info_and_show(stnet_imgs_folder+'/C/C1.jpg', stnet_st_postion + '/C1.csv')
display_image_info_and_show(stnet_imgs_folder+'/C/C2.jpg', stnet_st_postion + '/C2.csv')
display_image_info_and_show(stnet_imgs_folder+'/C/C3.jpg', stnet_st_postion + '/C3.csv')

In [ ]:
registration_xfeat_her2st_dir = []

registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/A/registration_xfeat')


print(registration_xfeat_her2st_dir)
from PIL import ImageDraw,Image
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_rectangles_on_images(all_img_files, what_can_i_say, rect_size=224):
    num_colors = len(what_can_i_say)
    colors = plt.cm.get_cmap('tab10', num_colors) 
    for i, img_file in enumerate(all_img_files):
        img = Image.open(img_file)
        draw = ImageDraw.Draw(img)
        for j, ((src_x, src_y), dsts_aff) in enumerate(what_can_i_say):
            if i == 0:
                x, y = src_x, src_y
            else:
                x, y = dsts_aff[i - 1]
            rect_x1 = x - rect_size / 2
            rect_y1 = y - rect_size / 2
            rect_x2 = x + rect_size / 2
            rect_y2 = y + rect_size / 2
            color_rgb = tuple(int(255 * c) for c in colors(j)[:3])
            draw.rectangle([rect_x1, rect_y1, rect_x2, rect_y2], outline=color_rgb, width=20)
        plt.figure(figsize=(10, 10))
        plt.imshow(img)
        plt.title(f"Image {i+1}")
        plt.axis('off')
        plt.show()

def plot_aff_spots(src_img_file,src_spot_file,trans_aff,dstes_img_files,num_spots=10):
    print("--------------------  Lets polt something ------------------------")
    img_nums = len(dstes_img_files)+1
    assert(img_nums-1 == len(dstes_img_files))
    assert(img_nums-1 == len(trans_aff))
    src_spots = pd.read_csv(src_spot_file)
    what_can_i_say = []
    for i in range(num_spots):
        src_x = src_spots['pixel_x'].iloc[i]
        src_y = src_spots['pixel_y'].iloc[i]
        current_coords = np.array([src_x, src_y, 1])
        dsts_aff = []
        for j in range(len(trans_aff)):
            aff_matrix = np.load(trans_aff[j])
            aff_matrix = np.vstack([aff_matrix, [0, 0, 1]])
            inv_aff_matrix = np.linalg.inv(aff_matrix)
            transformed_coord = inv_aff_matrix @ current_coords
            dst_x = transformed_coord[0]
            dst_y = transformed_coord[1]
            current_coords = transformed_coord
            dsts_aff.append((dst_x,dst_y))
        what_can_i_say.append(((src_x,src_y),dsts_aff))
    print(f"{num_spots}个坐标进行变换结果是：{what_can_i_say}")
    assert(len(what_can_i_say)==num_spots)
    all_img_files = [src_img_file] + dstes_img_files
    draw_rectangles_on_images(all_img_files, what_can_i_say)
    
sample_spots_nums = [297]
index = 0

Image.MAX_IMAGE_PIXELS = None 
for sample in registration_xfeat_her2st_dir:
    print(sample)
    orders = []
    trans_aff = []
    for order in os.listdir(sample):
        orders.append(order)
        for file in os.listdir(os.path.join(sample,order)):
            if file.endswith('.npy'):
                trans_aff.append(os.path.join(sample,order,file))
    print(orders)
    print(trans_aff)
    assert(len(orders) == len(trans_aff))
    nums = len(orders)
    src_img = orders[0].split('_')[2]
    print(src_img)
    dstes = []
    for i in orders:
        dstes.append(i.split('_')[0])
    print(dstes)
    src_img_file = os.path.join(os.path.dirname(sample),f'{src_img}.jpg')
    src_spot_file = os.path.join(stnet_st_postion,f'{src_img}.csv')
    dstes_img_files = []
    for i in dstes:
        dstes_img_files.append(os.path.join(os.path.dirname(sample),f'{i}.jpg'))
    print(dstes_img_files)
    plot_aff_spots(src_img_file,src_spot_file,trans_aff,dstes_img_files,num_spots=sample_spots_nums[index])
    index += 1
    break
    

In [ ]:
registration_xfeat_her2st_dir = []
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/A/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/B/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/C/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/D/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/E/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/F/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/G/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/H/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/I/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/J/registration_xfeat')
registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/K/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/L/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/M/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/N/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/O/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/P/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/Q/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/R/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/S/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/T/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/U/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/V/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/W/registration_xfeat')

import os
import numpy as np
import pandas as pd

block_size = 224

def transform_coordinates(src_x, src_y, aff_matrix):
    src_coords = np.array([src_x, src_y, 1])
    if aff_matrix.shape == (2, 3):
        aff_matrix = np.vstack([aff_matrix, [0, 0, 1]])
    inv_aff_matrix = np.linalg.inv(aff_matrix)
    dst_coords = np.dot(inv_aff_matrix, src_coords)
    return dst_coords[0], dst_coords[1]

def get_surrounding_spots(x, y, spots_df):
    surrounding_spots = []
    matched_spot = spots_df[(spots_df['x'] == x) & (spots_df['y'] == y)]
    if not matched_spot.empty:
        surrounding_spots.append(matched_spot.index[0]) 
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0:
                continue  
            
            matched_spot = spots_df[(spots_df['x'] == x + dx) & (spots_df['y'] == y + dy)]
            if not matched_spot.empty:
                surrounding_spots.append(matched_spot.index[0])
    return surrounding_spots

def save_mapping_to_csv(mapping_results, output_file):
    df = pd.DataFrame(mapping_results)
    df.to_csv(output_file, index=False)
    print(f"Mapping results saved to: {output_file}")


def find_nearest_spot(dst_x, dst_y, spots_df, image_name):
    distances = np.sqrt((spots_df['pixel_x'] - dst_x)**2 + (spots_df['pixel_y'] - dst_y)**2)
    nearest_index = distances.idxmin()
    nearest_pixel_x = spots_df.loc[nearest_index]['pixel_x']
    nearest_pixel_y = spots_df.loc[nearest_index]['pixel_y']
    if abs(nearest_pixel_x - dst_x) > 112 and abs(nearest_pixel_y - dst_y) > 112 and nearest_index > 180:
        print(f"Image '{image_name}': Nearest spot is out of range: ({nearest_pixel_x}, {nearest_pixel_y})")
        return 0, 0
    nearest_block_x = spots_df.loc[nearest_index]['x']
    nearest_block_y = spots_df.loc[nearest_index]['y']
    
    return nearest_block_x, nearest_block_y

def get_subdirectories(directory):
    return [name for name in os.listdir(directory) if os.path.isdir(os.path.join(directory, name))]

for sample_dir in registration_xfeat_her2st_dir:
    print(f"Processing sample: {sample_dir}")
    sample_name = os.path.basename(os.path.dirname(sample_dir))
    print(f"Sample name: {sample_name}")
    mapping_results = []
    orders = []
    trans_aff = []
    subdirs = get_subdirectories(sample_dir)
    for order in subdirs:
        orders.append(order)
        for file in os.listdir(os.path.join(sample_dir, order)):
            if file.endswith('.npy'):
                trans_aff.append(os.path.join(sample_dir, order, file))
    print(f"Orders: {orders}")
    print(f"Affine files: {trans_aff}")
    assert len(orders) == len(trans_aff)
    src_img = orders[0].split('_')[2]
    print(f"Source image: {src_img}")
    dstes = [order.split('_')[0] for order in orders]
    print(f"Target images: {dstes}")
    src_spot_file = os.path.join(stnet_st_postion, f'{src_img}.csv')
    src_spots = pd.read_csv(src_spot_file)
    dst_spots_files = [os.path.join(stnet_st_postion, f'{dst}.csv') for dst in dstes]
    dst_spots_dfs = [pd.read_csv(file) for file in dst_spots_files]
    
    for spot_index in range(len(src_spots)):
        src_x = src_spots['pixel_x'].iloc[spot_index]
        src_y = src_spots['pixel_y'].iloc[spot_index]
        src_block_x = src_spots['x'].iloc[spot_index]
        src_block_y = src_spots['y'].iloc[spot_index]
        src_surrounding_spots = get_surrounding_spots(src_block_x, src_block_y, src_spots)
        
        spot_mapping = {f'{src_img}_Spots': src_surrounding_spots}
        current_x, current_y = src_x, src_y
        for j, dst in enumerate(dstes):
            selc_path = os.path.join(stnet_st_postion,dst+'.csv')
            aff_matrix = np.load(trans_aff[j])
            dst_x, dst_y = transform_coordinates(current_x, current_y, aff_matrix)
            current_x, current_y = dst_x, dst_y
            dst_sport_dfs = pd.read_csv(selc_path)
            nearest_block_x,nearest_block_y = find_nearest_spot(dst_x,dst_y,dst_sport_dfs,dst)
            if nearest_block_x != 0 and nearest_block_y != 0:
                dst_block_x = nearest_block_x
                dst_block_y = nearest_block_y
                dst_surrounding_spots = get_surrounding_spots(dst_block_x, dst_block_y, dst_spots_dfs[j])
                spot_mapping[f'{dst}_Spots'] = dst_surrounding_spots
            else:
                spot_mapping[f'{dst}_Spots'] = []
        mapping_results.append(spot_mapping)
    os.makedirs('./stnet_mappings',exist_ok=True)
    output_file = os.path.join('./stnet_mappings/', f'{sample_name}_spot_mapping.csv')
    save_mapping_to_csv(mapping_results, output_file)
    mapping_results = []

In [ ]:
import os
import pandas as pd

folder = stnet_st_postion
gene_folder = stnet_gene_expression

for file in os.listdir(folder):
    print(f"Processing file: {file}")
    file_path = os.path.join(folder, file)
    df_csv = pd.read_csv(file_path)
    xy_strings = [f"{x}x{y}" for x, y in zip(df_csv['x'], df_csv['y'])]
    tsv_file_path = os.path.join(gene_folder, file.replace('.csv', '.tsv'))
    
    if not os.path.exists(tsv_file_path):
        print(f"Corresponding TSV file not found for {file}")
        continue
    with open(tsv_file_path, 'r') as tsv_file:
        lines = tsv_file.readlines()
        
        missing_xy_headers = []
        
        for i, xy_string in enumerate(xy_strings):
            if not any(line.startswith(xy_string) for line in lines):
                missing_xy_headers.append(i)
        
        if missing_xy_headers:
            print(f"No matching headers found in {tsv_file_path} for the following XY combinations:")
            print([xy_strings[i] for i in missing_xy_headers])
            df_csv.drop(missing_xy_headers, inplace=True)
            df_csv.to_csv(file_path, index=False)
            print(f"Updated CSV saved to {file_path}")


In [ ]:
def find_nearest_spot(dst_x, dst_y, spots_df, image_name):
    distances = np.sqrt((spots_df['pixel_x'] - dst_x)**2 + (spots_df['pixel_y'] - dst_y)**2)
    nearest_index = distances.idxmin()
    nearest_pixel_x = spots_df.loc[nearest_index]['pixel_x']
    nearest_pixel_y = spots_df.loc[nearest_index]['pixel_y']
    nearest_block_x = spots_df.loc[nearest_index]['x']
    nearest_block_y = spots_df.loc[nearest_index]['y']
    
    return int(nearest_block_x), int(nearest_block_y),nearest_pixel_x,nearest_pixel_y

def get_surrounding_spots(x, y, spots_df, pixel_x, pixel_y, name):
    if name in ['H2', 'G2', 'Q1']:
        num_neighbors = 12
    elif name in ['H3', 'G3', 'Q2']:
        num_neighbors = 13
    else:
        num_neighbors = 8
    surrounding_spots = []
    matched_spot = spots_df[(spots_df['x'] == x) & (spots_df['y'] == y)]
    if not matched_spot.empty:
        surrounding_spots.append(int(matched_spot.index[0])) 
    
    distances = np.sqrt((spots_df['pixel_x'] - pixel_x)**2 + (spots_df['pixel_y'] - pixel_y)**2)
    closest_indices = distances.nsmallest(num_neighbors + 1).index.tolist()
    
    for idx in closest_indices:
        if idx not in surrounding_spots:
            surrounding_spots.append(idx)
        if len(surrounding_spots) >= num_neighbors + 1:
            break
    
    return surrounding_spots

for sample_dir in registration_xfeat_her2st_dir:
    print(f"Processing sample: {sample_dir}")
    sample_name = os.path.basename(os.path.dirname(sample_dir))
    if sample_name != 'K':
        continue
    print(f"Sample name: {sample_name}")
    mapping_results = []
    orders = []
    trans_aff = []
    subdirs = get_subdirectories(sample_dir)
    for order in subdirs:
        orders.append(order)
        for file in os.listdir(os.path.join(sample_dir, order)):
            if file.endswith('.npy'):
                trans_aff.append(os.path.join(sample_dir, order, file))
    print(f"Orders: {orders}")
    print(f"Affine files: {trans_aff}")
    assert len(orders) == len(trans_aff)
    src_img = orders[0].split('_')[2]
    print(f"Source image: {src_img}")
    dstes = [order.split('_')[0] for order in orders]
    print(f"Target images: {dstes}")
    src_spot_file = os.path.join(stnet_st_postion, f'{src_img}.csv')
    src_spots = pd.read_csv(src_spot_file)
    dst_spots_files = [os.path.join(stnet_st_postion, f'{dst}.csv') for dst in dstes]
    dst_spots_dfs = [pd.read_csv(file) for file in dst_spots_files]
    
    
    for spot_index in range(len(src_spots)):
        src_name = src_img
        src_x = src_spots['pixel_x'].iloc[spot_index]
        src_y = src_spots['pixel_y'].iloc[spot_index]
        src_block_x = src_spots['x'].iloc[spot_index]
        src_block_y = src_spots['y'].iloc[spot_index]
        src_surrounding_spots = get_surrounding_spots(src_block_x, src_block_y, src_spots, src_x, src_y,src_name)
        
        spot_mapping = {f'{src_img}_Spots': src_surrounding_spots}
        current_x, current_y = src_x, src_y
        for j, dst in enumerate(dstes):
            selc_path = os.path.join(stnet_st_postion,dst+'.csv')
            aff_matrix = np.load(trans_aff[j])
            dst_x, dst_y = transform_coordinates(current_x, current_y, aff_matrix)
            current_x, current_y = dst_x, dst_y
            dst_sport_dfs = pd.read_csv(selc_path)
            nearest_block_x,nearest_block_y,nearest_pixel_x,nearest_pixel_y = find_nearest_spot(dst_x,dst_y,dst_sport_dfs,dst)
            dst_block_x = nearest_block_x
            dst_block_y = nearest_block_y
            
            dst_surrounding_spots = get_surrounding_spots(dst_block_x, dst_block_y, dst_spots_dfs[j],nearest_pixel_x,nearest_pixel_y,dst)
            spot_mapping[f'{dst}_Spots'] = dst_surrounding_spots
        mapping_results.append(spot_mapping)
    
    os.makedirs('./stnet_mappings',exist_ok=True)
    output_file = os.path.join('./stnet_mappings/', f'{sample_name}_spot_mapping.csv')
    save_mapping_to_csv(mapping_results, output_file)
    mapping_results = []
    

In [2]:
registration_xfeat_her2st_dir = []
registration_xfeat_her2st_dir.append(skin_imgs_folder + '/A/registration_xfeat')
# registration_xfeat_her2st_dir.append(skin_imgs_folder + '/B/registration_xfeat')
# registration_xfeat_her2st_dir.append(skin_imgs_folder + '/C/registration_xfeat')
# registration_xfeat_her2st_dir.append(skin_imgs_folder + '/D/registration_xfeat')

In [ ]:
import os
import pandas as pd
import numpy as np

def find_nearest_spot(dst_x, dst_y, spots_df, image_name):
    distances = np.sqrt((spots_df['pixel_x'] - dst_x)**2 + (spots_df['pixel_y'] - dst_y)**2)
    nearest_index = distances.idxmin()
    nearest_pixel_x = spots_df.loc[nearest_index]['pixel_x']
    nearest_pixel_y = spots_df.loc[nearest_index]['pixel_y']
    nearest_block_x = spots_df.loc[nearest_index]['x']
    nearest_block_y = spots_df.loc[nearest_index]['y']
    
    return int(nearest_block_x), int(nearest_block_y),nearest_pixel_x,nearest_pixel_y

def get_surrounding_spots(x, y, spots_df, pixel_x, pixel_y, name):
    if name in ['H2', 'G2', 'Q1']:
        num_neighbors = 12
    elif name in ['H3', 'G3', 'Q2']:
        num_neighbors = 13
    else:
        num_neighbors = 8
    surrounding_spots = []
    matched_spot = spots_df[(spots_df['x'] == x) & (spots_df['y'] == y)]
    if not matched_spot.empty:
        surrounding_spots.append(int(matched_spot.index[0])) 
    
    distances = np.sqrt((spots_df['pixel_x'] - pixel_x)**2 + (spots_df['pixel_y'] - pixel_y)**2)
    closest_indices = distances.nsmallest(num_neighbors + 1).index.tolist()
    
    for idx in closest_indices:
        if idx not in surrounding_spots:
            surrounding_spots.append(idx)
        if len(surrounding_spots) >= num_neighbors + 1:
            break
    
    assert(len(surrounding_spots) == 9)
    
    return surrounding_spots

def transform_coordinates(src_x, src_y, aff_matrix):
    src_coords = np.array([src_x, src_y, 1])
    if aff_matrix.shape == (2, 3):
        aff_matrix = np.vstack([aff_matrix, [0, 0, 1]])
    inv_aff_matrix = np.linalg.inv(aff_matrix)
    dst_coords = np.dot(inv_aff_matrix, src_coords)
    return dst_coords[0], dst_coords[1]

def get_subdirectories(directory):
    return [name for name in os.listdir(directory) if os.path.isdir(os.path.join(directory, name))]

def save_mapping_to_csv(mapping_results, output_file):
    df = pd.DataFrame(mapping_results)
    df.to_csv(output_file, index=False)
    print(f"Mapping results saved to: {output_file}")
    
for sample_dir in registration_xfeat_her2st_dir:
    print(f"Processing sample: {sample_dir}")
    
    sample_name = os.path.basename(os.path.dirname(sample_dir))
    print(f"Sample name: {sample_name}")
    mapping_results = []
    orders = []
    trans_aff = []
    subdirs = get_subdirectories(sample_dir)
    for order in subdirs:
        orders.append(order)
        for file in os.listdir(os.path.join(sample_dir, order)):
            if file.endswith('.npy'):
                trans_aff.append(os.path.join(sample_dir, order, file))
    print(f"Orders: {orders}")
    print(f"Affine files: {trans_aff}")
    assert len(orders) == len(trans_aff)
    src_img = orders[0].split('_')[2]
    print(f"Source image: {src_img}")
    
    dstes = [order.split('_')[0] for order in orders]
    print(f"Target images: {dstes}")
    
    src_spot_file = os.path.join(skin_st_postion, f'{src_img}.tsv')
    src_spots = pd.read_csv(src_spot_file,sep='\t')
    
    dst_spots_files = [os.path.join(skin_st_postion, f'{dst}.tsv') for dst in dstes]
    dst_spots_dfs = [pd.read_csv(file,sep='\t') for file in dst_spots_files]
    
    
    for spot_index in range(len(src_spots)):
        src_name = src_img
        src_x = src_spots['pixel_x'].iloc[spot_index]
        src_y = src_spots['pixel_y'].iloc[spot_index]
        src_block_x = src_spots['x'].iloc[spot_index]
        src_block_y = src_spots['y'].iloc[spot_index]
        src_surrounding_spots = get_surrounding_spots(src_block_x, src_block_y, src_spots, src_x, src_y,src_name)
        
        spot_mapping = {f'{src_img}_Spots': src_surrounding_spots}
        current_x, current_y = src_x, src_y
        for j, dst in enumerate(dstes):
            selc_path = os.path.join(skin_st_postion,dst+'.tsv')
            aff_matrix = np.load(trans_aff[j])
            dst_x, dst_y = transform_coordinates(current_x, current_y, aff_matrix)
            current_x, current_y = dst_x, dst_y
            dst_sport_dfs = pd.read_csv(selc_path,sep='\t')
            nearest_block_x,nearest_block_y,nearest_pixel_x,nearest_pixel_y = find_nearest_spot(dst_x,dst_y,dst_sport_dfs,dst)
            dst_block_x = nearest_block_x
            dst_block_y = nearest_block_y
            
            dst_surrounding_spots = get_surrounding_spots(dst_block_x, dst_block_y, dst_spots_dfs[j],nearest_pixel_x,nearest_pixel_y,dst)
            spot_mapping[f'{dst}_Spots'] = dst_surrounding_spots
            
        mapping_results.append(spot_mapping)

    
    os.makedirs('./skin_mappings',exist_ok=True)
    output_file = os.path.join('./skin_mappings/', f'{sample_name}_spot_mapping.csv')
    save_mapping_to_csv(mapping_results, output_file)
    mapping_results = []
    

In [ ]:
import pandas as pd
import glob

csv_files = glob.glob('../data/ST-images/PCW/ST-spotfiles/*.csv', recursive=True)

for file in csv_files:
    try:
        df = pd.read_csv(file, sep='\t', header=None)
        df.columns = ['pixel_y', 'pixel_x', 'r', 'x', 'y']
        df = df[['x', 'y', 'pixel_x', 'pixel_y', 'r']]
        df.to_csv(file, index=False)
        print(f"{file}")
    except Exception as e:
        print(f"file：{e}")